# Experiment

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import gc
import os
import pathlib
import subprocess
import sys
import pickle
import shutil
import json
import glob
import re
import tempfile
from pathlib import Path
from typing import Iterable
import anndata as ad
import copy

docs_dir = str(pathlib.Path(os.getcwd()).resolve().parents[0])
sys.path.append(docs_dir)
print(docs_dir)

from src.install_cellranger import install_cellranger
from src.download_references import download_references
from src.build_cellranger_mkref import build_cellranger_mkref
from src.fastq_datasets import fastq_datasets
from src.download_fastq import download_fastq
from src.run_cellranger_master import existing_outs_for_release
from src.anndata_generator import anndata_generator

/ictstr01/home/icb/kemal.inecik/work/codes/idtrack/docs/_notebooks


In [4]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

import matplotlib.pyplot as plt
import seaborn as sns

# Important to have consistent figures across platforms:
_rcparams_path = os.path.join(docs_dir, "figure_rcparams", "rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

# Preparation

## Install cellranger

In [5]:
cellranger_params = dict(
    version="9.0.1",
    expected_md5="2efec98bff01f7a59edaf43724fae13f",
    url="https://cf.10xgenomics.com/releases/cell-exp/cellranger-9.0.1.tar.gz?Expires=1756836323&Key-Pair-Id=APKAI7S6A5RYOXBWRPDA&Signature=kjLqu7Mer5A0F4hV1PlJQXFGoY0pJZC8OsPilkGkIBOdxG~crQI5eLMSvV~cZzx0u94PrhZvlXofddsjzhvFXe3mPd2bauIyP5RxITf7QgpHWeZ6hNFhLI9CD5jJXgGs0bzhNhSEquxlLjHH~W3v7zpwk5mzV7jgD7tCwd00C4q-yRtwNefer~ZV7p01FwOIlo6dckef7dMgDIb~6FBuGlkDbyChbJOWITXSGyxz7IwRkFIt9C4X2bsPQ29n3I0gX3zVIAbavgKVC0ZOk0lUx6SucGUQZbhQeTvLysWDhRldj7TrYn9ZoJpo8fz50JMSTn9o3QNatz0rBs-DQasyNg__",
    apps_dir=pathlib.Path("/home/icb/kemal.inecik/tools/apps").expanduser(),
    tmp_dir=pathlib.Path("/home/icb/kemal.inecik/tools/tmp").expanduser(),
)

In [5]:
cellranger_bin = install_cellranger(**cellranger_params)

[cellranger] Downloading with curl to /ictstr01/home/icb/kemal.inecik/tools/tmp/cellranger-9.0.1.tar.gz ...
[cellranger] MD5 (python) = 2efec98bff01f7a59edaf43724fae13f
[cellranger] MD5 (md5sum) = 2efec98bff01f7a59edaf43724fae13f
[cellranger] Extracting with tar to /ictstr01/home/icb/kemal.inecik/tools/apps ...
[cellranger] Done. Installed at: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1
[cellranger] Tarball kept at: /ictstr01/home/icb/kemal.inecik/tools/tmp/cellranger-9.0.1.tar.gz


In [6]:
cellranger_bin = install_cellranger(**cellranger_params)
os.environ["PATH"] = f"{cellranger_bin.parent}:{os.environ['PATH']}"
print("bin:", cellranger_bin)
print("version:", subprocess.run([str(cellranger_bin), "--version"], text=True, capture_output=True).stdout)

[cellranger] Found existing install: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/cellranger
bin: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/cellranger
version: cellranger cellranger-9.0.1



## Download annotation sources

In [7]:
download_references_params = dict(
    workdir = pathlib.Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard"),
    releases = range(80, 115),  # 80..114 inclusive
    fasta_name = "Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz",
    gunzip=True,        
    keep_gz=True,       
)

In [7]:
paths_references = download_references(silent=False, **download_references_params)

[release 80] Downloaded: Homo_sapiens.GRCh38.80.gtf.gz
[release 80] Gunzipped: Homo_sapiens.GRCh38.80.gtf
[release 80] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 80] Renamed: Homo_sapiens.GRCh38.80.dna.primary_assembly.fa.gz
[release 80] Gunzipped: Homo_sapiens.GRCh38.80.dna.primary_assembly.fa
[release 81] Downloaded: Homo_sapiens.GRCh38.81.gtf.gz
[release 81] Gunzipped: Homo_sapiens.GRCh38.81.gtf
[release 81] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 81] Renamed: Homo_sapiens.GRCh38.81.dna.primary_assembly.fa.gz
[release 81] Gunzipped: Homo_sapiens.GRCh38.81.dna.primary_assembly.fa
[release 82] Downloaded: Homo_sapiens.GRCh38.82.gtf.gz
[release 82] Gunzipped: Homo_sapiens.GRCh38.82.gtf
[release 82] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 82] Renamed: Homo_sapiens.GRCh38.82.dna.primary_assembly.fa.gz
[release 82] Gunzipped: Homo_sapiens.GRCh38.82.dna.primary_assembly.fa
[release 83] Downloaded: Homo_sapiens

In [8]:
paths_references = download_references(silent=True, **download_references_params)

## Create reference for cellranger

In [9]:
build_cellranger_mkref_params = dict(
    paths_by_release = paths_references,
    cellranger_bin = cellranger_bin,
    assembly_label = "GRCh38",
    use_threads = 28
)

In [10]:
paths_cellranger_mkref = build_cellranger_mkref(silent=False, **build_cellranger_mkref_params)

[release 80] Running: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/bin/cellranger mkgtf /lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.gtf /lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.filtered.gtf --attribute=gene_biotype:protein_coding --attribute=gene_type:protein_coding
[release 80] mkgtf OK: Homo_sapiens.GRCh38.80.filtered.gtf
[release 80] Running: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/bin/cellranger mkref --genome=reference_GRCh38_80 --fasta=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.dna.primary_assembly.fa --genes=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.filtered.gtf --nthreads=28 (cwd=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_exper

In [10]:
paths_cellranger_mkref = build_cellranger_mkref(silent=True, **build_cellranger_mkref_params)

## Download sequencing data

In [11]:
download_fastq_params = dict(
    datasets = fastq_datasets,
    working_dir = pathlib.Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/fastq_raw"),
)

In [ ]:
paths_download_fastq = download_fastq(silent=False, **download_fastq_params)

Downloading: 'pbmc_1k_v3'
Downloading: 'pbmc_20k_donors1_4_multiplex_gemx_3p'
Downloading: 'pbmc_10k_5p_v3_ultima'
Downloading: 'pbmc_10k_3p_v31_si'


In [12]:
paths_download_fastq = download_fastq(silent=True, **download_fastq_params)

## Run alignments

In [13]:
override = False
datasets = ["pbmc_1k_v3"]  # you can add more later
assembly_labels = ["GRCh38"]

prior_workdir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments").resolve()
working_dir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments").resolve()
working_dir.mkdir(parents=True, exist_ok=True)
run_cellranger_script = Path.cwd().resolve().parent / "src" / "run_cellranger_master.py"

count = 0
errors = []

for dataset_key in paths_download_fastq:

    if dataset_key not in datasets:
        continue
    
    # (A) discover releases for the first (and only) assembly in your note, but we allow multiple
    for assembly_label in assembly_labels:
        
        for release, release_content in paths_cellranger_mkref.items():

            # if count > 0:
            #     break

            if release_content["status"] != 'done':
                print(f"[WARN] No releases discovered for assembly {assembly_label!r}, release {release!r}.")
                continue
            
            # Per your request: create a folder for each dataset / release / assembly under working_dir
            job_dir = (working_dir / dataset_key / f"{assembly_label}_{release}").resolve()
            job_dir.mkdir(parents=True, exist_ok=True)
            # Slurm log lives in the "respective" folder
            slurm_log = job_dir / f"slurm_job.log"
            # Skip if existing output (unless override)
            existing = existing_outs_for_release(working_dir, dataset_key, assembly_label, release)
            if existing and not override:
                print(f"[SKIP] Found existing metrics for {dataset_key} release {release}: {existing}")
                continue

            # Build Slurm script (kept inside the notebook cell, per your example)
            slurm_script = f"""#!/bin/bash
#SBATCH -J cr_{dataset_key}_r{release}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 4
#SBATCH --mem=32G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {slurm_log}
#SBATCH -e {slurm_log}

echo "Start"

set -eo pipefail

echo "Environment setup"
source ~/.bashrc && conda activate idtrack_dev_env

echo "Python started"
python -u "{run_cellranger_script}" \\
  --dataset-key "{dataset_key}" \\
  --release {release} \\
  --workdir "{prior_workdir}" \\
  --results-dir "{working_dir}" \\
  --assembly-label "{assembly_label}" \\
  --cellranger-bin "{cellranger_bin}" \\
  --no-download

echo "End."
"""
            try:
                script_name = job_dir / f"slurm_job.sh"
                with open(script_name, "w") as f:
                    f.write(slurm_script)
    
                print(f"Submitting: {dataset_key} / {assembly_label} / r{release} / log {slurm_log}")
                subprocess.run(["sbatch", str(script_name)], check=False)
                count += 1
        
            finally:
                # os.remove(script_name)
                pass
                
print(f" - Number of jobs submitted: {count}")

[WARN] No releases discovered for assembly 'GRCh38', release 80.
[WARN] No releases discovered for assembly 'GRCh38', release 81.
Submitting: pbmc_1k_v3 / GRCh38 / r82 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_82/slurm_job.log
Submitted batch job 30744789
Submitting: pbmc_1k_v3 / GRCh38 / r83 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_83/slurm_job.log
Submitted batch job 30744790
Submitting: pbmc_1k_v3 / GRCh38 / r84 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_84/slurm_job.log
Submitted batch job 30744791
Submitting: pbmc_1k_v3 / GRCh38 / r85 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_85/slurm_job.log
Submitted batch job 30744792
Submitting: pbmc_1k_v3 / GRCh38 / r86 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRC

## Prepare anndata

In [13]:
anndata_generator_params = dict(
    alignments_root=Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments"),
    out_root="/home/icb/kemal.inecik/lustre_workspace/idtrack_experiments/anndatas",
    keep_obs_if_it_is_filtered_by_cellranger=True
)

In [14]:
anndata_paths = anndata_generator(silent=False, **anndata_generator_params)

[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_100/pbmc_1k_v3_count_r100/outs (run=pbmc_1k_v3_count_r100) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_100.h5ad  shape=1221x19970
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_101/pbmc_1k_v3_count_r101/outs (run=pbmc_1k_v3_count_r101) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_101.h5ad  shape=1221x19966
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_102/pbmc_1k_v3_count_r102/outs (run=pbmc_1k_v3_count_r102) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_102.h5ad  shape=1222x19973
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_103/pbmc_1k_v3_count_r103/outs (run=pbmc_1k_v3_count_r103) [filtered_only=True]
[OK] Wrote pbmc_1k_v

In [14]:
datasets = ["pbmc_1k_v3"]
assembly_labels = ["GRCh38"]
anndata_paths = anndata_generator(silent=True, **anndata_generator_params)

In [15]:
adata = ad.read_h5ad(anndata_paths[datasets[0]][assembly_labels[0]]['100'])
adata

AnnData object with n_obs × n_vars = 1221 × 19970
    obs: 'is_filtered_by_cellranger'
    var: 'gene_id', 'gene_symbol', 'feature_type', 'genome'
    uns: 'cellranger', 'provenance'

# Analysis

## Get the IDs

In [16]:
adata_dict = copy.deepcopy(anndata_paths)
for _ap in anndata_paths:
    for _al in assembly_labels:
        for _er in anndata_paths[_ap][_al]:
            adata_dict[_ap][_al][_er] = ad.read_h5ad(anndata_paths[_ap][_al][_er])

### Before running IDTrack (important)

This notebook is the **gold-standard data generation** scaffold. It intentionally separates *data creation* from *manuscript analyses*.

For the manuscript-facing results and figures, run:

- `analysis_gold_standard_fig4c.ipynb` (Figure 4c + feature-set consistency)
- `analysis_gold_standard_expression_consistency.ipynb` (pseudo-bulk consistency marketing appendix)

Cache discipline:

- Use the shared cache under `idtrack/docs/_notebooks/idtrack_cache/` by setting `IDTRACK_LOCAL_REPO` (recommended).
- The analysis notebooks are cache-first and will reuse the conversion outputs generated here.


## Running IDTrack

### Test run

In [79]:
sys.path.append("/home/icb/kemal.inecik/work/codes/idtrack")
import idtrack

idtrack_local_dir = "/lustre/groups/ml01/workspace/kemal.inecik/idtrack_temp"
idtrack_project_local_repository = "/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/idtrack_runs"
os.makedirs(idtrack_project_local_repository, exist_ok=True)

idt = idtrack.API(local_repository=idtrack_local_dir)
idt.configure_logger()
organism_formal_name, _ = idt.resolve_organism("homo sapiens")
idt.build_graph(organism_name=organism_formal_name, snapshot_release=114, calculate_caches=False)
idt.calculate_graph_caches()

query_symbols = adata_dict[_ap][_al][_er].var["gene_symbol"].to_list()

2025-11-14 13:20:37 INFO:verify_organism: Ensembl Rest API query to get the organism names and associated releases.
2025-11-14 13:20:37 INFO:graph_maker: The graph is being read: /lustre/groups/ml01/workspace/kemal.inecik/idtrack_temp/graph_homo_sapiens_min79_max114_narrow.pickle
2025-11-14 13:21:46 INFO:the_graph: Cached properties being calculated: combined_edges
2025-11-14 13:22:58 INFO:the_graph: Cached properties being calculated: combined_edges_assembly_specific_genes
2025-11-14 13:23:01 INFO:the_graph: Cached properties being calculated: combined_edges_genes
2025-11-14 13:23:12 INFO:the_graph: Cached properties being calculated: lower_chars_graph
2025-11-14 13:23:13 INFO:the_graph: Cached properties being calculated: get_active_ranges_of_id
2025-11-14 13:23:32 INFO:the_graph: Cached properties being calculated: available_external_databases
2025-11-14 13:23:33 INFO:the_graph: Cached properties being calculated: available_genome_assemblies
2025-11-14 13:23:33 INFO:the_graph: Cache

In [80]:
matched_symbols = idt.convert_identifier_multiple(query_symbols, to_release=114, final_database="HGNC Symbol")
matched_symbols_classified = idt.classify_multiple_conversion(matched_symbols)
idt.print_binned_conversion(matched_symbols_classified)

100%|██████████████████████████████████████████| 19968/19968 [03:48<00:00, 87.44it/s, ID:AC213203.1]
2025-11-14 13:27:45 INFO:api: 
IDTrack conversion summary:
  Total processed: 19968
  1→0: 18 (0.1%)
  1→1: 19950 (99.9%)
    Changed only: 755 (3.8%)
    Alternative targets: 470 (2.4%)
    Rest: 18725 (93.9%)
  1→n: 0 (0.0%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
  Diagnostics:
    no_corresponding: 0
    no_conversion:   18
    no_target:       470


### Batch run

In [111]:
override = False

prior_workdir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments").resolve()
working_dir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/idtrack_runs").resolve()
working_dir.mkdir(parents=True, exist_ok=True)
run_idtrack_script = Path.cwd().resolve().parent / "src" / "run_idtrack.py"

count = 0
errors = []

for dataset_key in anndata_paths:
    
    for assembly_label in anndata_paths[dataset_key]:
        
        for from_release, anndata_path in anndata_paths[dataset_key][assembly_label].items():
            
            for from_database in ['gene_id', 'gene_symbol']:

                for to_database in ["ensembl_gene", "HGNC Symbol"]:

                    # if count > 0:
                        #     break

                    to_release_str_list = '|'.join(map(str, sorted(map(int, list(anndata_paths[dataset_key][assembly_label].keys())))))
                    
                    # create a folder for each dataset / release / assembly under working_dir
                    job_dir = (working_dir / dataset_key / f"{assembly_label}_{from_release}").resolve()
                    job_dir.mkdir(parents=True, exist_ok=True)
                    
                    # Slurm log lives in the "respective" folder
                    count += 1
                    job_name = f"{from_database}__{to_database}__{to_release_str_list}"
                    slurm_job_name = f"idtrack_{count}"
                    slurm_log = job_dir / f"log_{count}.log"
                    result_pickle_path = os.path.join(job_dir, f"{job_name}.pickle")
                    
                    # Skip if existing output (unless override)
        
                    if os.path.isfile(result_pickle_path) and not override:
                        print(f"[SKIP] Found existing: {slurm_job_name}")
                        continue

                    # Build Slurm script (kept inside the notebook cell, per your example)
                    slurm_script = f"""#!/bin/bash
#SBATCH -J {slurm_job_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 1
#SBATCH --mem=60G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {slurm_log}
#SBATCH -e {slurm_log}

echo "Start"

set -eo pipefail

echo "Environment setup"
source ~/.bashrc && conda activate idtrack_dev_env

echo "Python started"
python -u "{run_idtrack_script}" \\
  --anndata-path "{anndata_path}" \\
  --dataset-key "{dataset_key}" \\
  --assembly-label "{assembly_label}" \\
  --from-release "{from_release}" \\
  --to-release-str-list "{to_release_str_list}" \\
  --from-database "{from_database}" \\
  --to-database "{to_database}" \\
  --slurm-job-name "{slurm_job_name}" \\
  --job-name "{job_name}" \\
  --job-dir "{job_dir}" 

echo "End."
"""
                    try:
                        script_name = job_dir / f"job_{count}.sh"
                        with open(script_name, "w") as f:
                            f.write(slurm_script)
            
                        print(f"Submitting: {slurm_job_name}")
                        subprocess.run(["sbatch", str(script_name)], check=False)
                    
                    finally:
                        # os.remove(script_name)
                        pass
                        
print(f" - Number of jobs submitted: {count}")

Submitting: idtrack_1
Submitted batch job 31469799
Submitting: idtrack_2
Submitted batch job 31469800
Submitting: idtrack_3
Submitted batch job 31469801
Submitting: idtrack_4
Submitted batch job 31469802
Submitting: idtrack_5
Submitted batch job 31469804
Submitting: idtrack_6
Submitted batch job 31469805
Submitting: idtrack_7
Submitted batch job 31469806
Submitting: idtrack_8
Submitted batch job 31469807
Submitting: idtrack_9
Submitted batch job 31469808
Submitting: idtrack_10
Submitted batch job 31469809
Submitting: idtrack_11
Submitted batch job 31469811
Submitting: idtrack_12
Submitted batch job 31469812
Submitting: idtrack_13
Submitted batch job 31469813
Submitting: idtrack_14
Submitted batch job 31469814
Submitting: idtrack_15
Submitted batch job 31469815
Submitting: idtrack_16
Submitted batch job 31469816
Submitting: idtrack_17
Submitted batch job 31469817
Submitting: idtrack_18
Submitted batch job 31469819
Submitting: idtrack_19
Submitted batch job 31469820
Submitting: idtrack_2

## Analysis of IDTrack performance

In [ ]:
# Collect the conversions
# Get adata_dict

In [18]:
pass

### Finalized figures

In [19]:
pass